In [27]:
import sys
import os
import json
import sqlite3
import json
import re
import nltk
import spacy
import subprocess
from jinja2 import Environment, FileSystemLoader
from dotenv import load_dotenv
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
load_dotenv()

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

spacy.prefer_gpu()
model = "en_core_web_md"
try:
    nlp = spacy.load(model)
except OSError:
    print(f"Downloading {model}...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", model])
    nlp = spacy.load(model)
    
ROOT_PATH = os.environ["ROOT_PATH"]
SPIDER_DB_PATH = f"{ROOT_PATH}/database/spider"

GOLD_DB = f"{ROOT_PATH}/database/gold/gold.sqlite"

def get_full_ddl(entry):
    full_ddl = json.loads(entry["full_ddl"])
    formatted_full_ddl = []
    for table in full_ddl:
        formatted_full_ddl.append(table)
    return "\n".join(formatted_full_ddl)

def get_simplified_ddl(entry):
    simplified_ddl = json.loads(entry["simplified_ddl"])
    formatted_simplified_ddl = []
    for table in simplified_ddl:
        formatted_simplified_ddl.append(table)
    return "\n".join(formatted_simplified_ddl)

def get_cell_values(entry, max_samples=3):
    db_id = entry["db_id"]
    db_path = os.path.join(SPIDER_DB_PATH, db_id, db_id + '.sqlite')
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]

    formatted_tables = []

    for table in tables:
        cursor.execute(f"PRAGMA table_info({table});")
        columns = [row[1] for row in cursor.fetchall()]  # row[1] is column name

        cursor.execute(f"SELECT * FROM {table} LIMIT {max_samples};")
        rows = cursor.fetchall()

        col_samples = list(zip(*rows)) if rows else [[] for _ in columns]

        col_strs = []
        for col, vals in zip(columns, col_samples):
            val_list = ", ".join(str(v) for v in vals[:max_samples])
            col_strs.append(f"{col}[{val_list}]")
        formatted = f"{table}(" + ", ".join(col_strs) + ")"
        formatted_tables.append(formatted)

    conn.close()
    return "\n".join(formatted_tables)

def get_foreign_keys(entry):
    foreign_keys = json.loads(entry["foreign_keys"])
    formatted_foreign_keys = []
    for fk in foreign_keys:
        formatted_foreign_keys.append(fk)
    return "\n".join(formatted_foreign_keys)

def create_prompts(entries, template, query_type="sql", skeleton_dataset=None):
    params = {}
    prompts = []
    for entry in entries:
        params["query_type"] = query_type
        params["question"] = entry["question"]
        params["full_ddl"] = get_full_ddl(entry)
        params["simplified_ddl"] = get_simplified_ddl(entry)
        params["foreign_keys"] = get_foreign_keys(entry)
        params["cell_values"] = get_cell_values(entry)
        params["few_shot"] = get_few_shot(entry["question"], entry["simplified_ddl"], skeleton_dataset)
        prompt = template.render(params)
        prompts.append(prompt)
    return prompts

def create_completions(entries, query_type="sql"):
    completions = []
    for entry in entries:
        completions.append(entry["query"] if query_type == "sql" else entry[query_type])
    return completions

def create_dataset(entries, template, strategy, skeleton_dataset=None):
    query_type = "sql" if strategy == "nl2SQL" else "natsql"
    processed_data = []
    prompts = create_prompts(entries, template, query_type, skeleton_dataset)
    completions = create_completions(entries, query_type)
    for prompt, completion in zip(prompts, completions):
        processed_data.append({"prompt": prompt, "completion": completion})
    return processed_data

def create_sql_dataset(entries):
    processed_data = []
    for entry in entries:
        processed_data.append(f'{entry["query"]}\t{entry["db_id"]}')
    return processed_data

def get_question_skeleton(question, schema):
    # Initialize lemmatizer
    lemmatizer = WordNetLemmatizer()
    
    # Parse the schema to extract table and column names
    try:
        schema_data = json.loads(schema)
    except json.JSONDecodeError:
        return question  # Return original if schema parsing fails
    
    # Extract all table names and column names from schema
    domain_tokens = set()
    
    for table_info in schema_data:
        # Extract table name (before the opening parenthesis)
        table_name = table_info.split('(')[0].strip()
        domain_tokens.add(table_name.lower())
        
        # Extract column names (inside parentheses)
        columns_part = table_info.split('(')[1].split(')')[0]
        columns = [col.strip().split()[0] for col in columns_part.split(',')]
        for col in columns:
            domain_tokens.add(col.lower())
    
    
    # Find and replace quoted strings (single or double quotes) with placeholders
    quoted_strings = []
    quote_pattern = r"'([^']*)'|\"([^\"]*)\""
    
    def replace_quoted(match):
        quoted_strings.append(match.group(0))
        return f"__QUOTED_STRING_{len(quoted_strings)-1}__"
    
    question_with_placeholders = re.sub(quote_pattern, replace_quoted, question)
    
    # Tokenize the question
    question_tokens = word_tokenize(question_with_placeholders.lower())
    
    # Create skeleton by replacing domain tokens with <mask>
    skeleton_tokens = []
    for token in question_tokens:
        # Check if it's a quoted string placeholder
        if token.startswith("__quoted_string_") and token.endswith("__"):
            skeleton_tokens.append('<mask>')
        
        # Check for numeric values
        elif token.isdigit() or (token.replace('.', '').replace(',', '').isdigit()):
            skeleton_tokens.append('<mask>')
        
        # Check for domain tokens (table/column names)
        else:
            lemmatized_token = lemmatizer.lemmatize(token)
            if lemmatized_token in domain_tokens or token in domain_tokens:
                skeleton_tokens.append('<mask>')
            else:
                skeleton_tokens.append(token)
    
    # Join tokens back into a sentence
    skeleton = ' '.join(skeleton_tokens)
    
    return skeleton

def create_skeleton_dataset(entries):
    processed_data = []
    for entry in entries:
        processed_data.append(f'{get_question_skeleton(entry["question"], entry["simplified_ddl"])}\t{entry["question"]}\t{entry["query"]}')
    return processed_data

def get_similar_skeletons(query_skeleton, train_skeleton_data, top_k=3):
    query_doc = nlp(query_skeleton)
    similarities = []
    for i, line in enumerate(train_skeleton_data):
        line = line.strip()
        skeleton_question, full_question, sql_query = line.split('\t', 2)
        similarity = query_doc.similarity(nlp(skeleton_question))
        similarities.append((i, similarity, skeleton_question, full_question, sql_query))
    
    # Sort by similarity (descending) and return top-k
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

def get_few_shot(question, schema, train_skeleton_data=None):
    few_shot = ""
    if train_skeleton_data is None:
        return few_shot
    else:
        skeleton_query = get_question_skeleton(question, schema)
        similar_skeletons = get_similar_skeletons(skeleton_query, train_skeleton_data, top_k=3)
        for i, (idx, score, skeleton_question, full_question, sql_query) in enumerate(similar_skeletons, 1):
            few_shot += f"{full_question}\n{sql_query}\n"
        return few_shot



[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/atissera/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [28]:
"""
Create dataset for either nl2SQL or nl2NatSQL models using specified template.

Args:
    strategy (str): Either 'nl2SQL' or 'nl2NatSQL'
    template_name (str): Template name (e.g., 'template_00' or 'template_00.j2')
    difficulties (list): Optional list of difficulty levels to filter by ('easy', 'medium', 'hard', 'extra')
    test_limit (int): Optional limit for number of test records to include
"""

strategy = 'nl2SQL'
template_name = 'template_00'
difficulties = ['easy']
test_limit = 10

if strategy not in ['nl2SQL', 'nl2NatSQL']:
    raise ValueError("strategy must be either 'nl2SQL' or 'nl2NatSQL'")

# Validate difficulties parameter
valid_difficulties = ['easy', 'medium', 'hard', 'extra']
if difficulties is not None:
    if not isinstance(difficulties, list):
        raise ValueError("difficulties must be a list")
    for difficulty in difficulties:
        if difficulty not in valid_difficulties:
            raise ValueError(f"Invalid difficulty '{difficulty}'. Must be one of: {valid_difficulties}")
    print(f"Filtering dataset by difficulties: {', '.join(difficulties)}")

# Validate test_limit parameter
if test_limit is not None:
    if not isinstance(test_limit, int) or test_limit <= 0:
        raise ValueError("test_limit must be a positive integer")
    print(f"Limiting test dataset to {test_limit} records")

# Connect to gold database
conn_gold = sqlite3.connect(GOLD_DB)
cursor_gold = conn_gold.cursor()

# Get table columns
table_columns = cursor_gold.execute("PRAGMA table_info(gold_dataset)").fetchall()
columns = [column[1] for column in table_columns]

# Build difficulty filter clause if difficulties are specified
difficulty_clause = ""
if difficulties is not None:
    placeholders = ",".join(["?" for _ in difficulties])
    difficulty_clause = f" AND difficulty IN ({placeholders})"

# Get train, valid, and test entries
train_query = f"SELECT {', '.join(columns)} FROM gold_dataset WHERE source = 'train'{difficulty_clause}"
if difficulties is not None:
    cursor_gold.execute(train_query, difficulties)
else:
    cursor_gold.execute(train_query)
train_rows = cursor_gold.fetchall()
train_entries = [dict(zip(columns, row)) for row in train_rows]

valid_query = f"SELECT {', '.join(columns)} FROM gold_dataset WHERE source = 'dev'{difficulty_clause}"
if difficulties is not None:
    cursor_gold.execute(valid_query, difficulties)
else:
    cursor_gold.execute(valid_query)
valid_rows = cursor_gold.fetchall()
valid_entries = [dict(zip(columns, row)) for row in valid_rows]

test_query = f"SELECT {', '.join(columns)} FROM gold_dataset WHERE source = 'test'{difficulty_clause}"
if difficulties is not None:
    cursor_gold.execute(test_query, difficulties)
else:
    cursor_gold.execute(test_query)
test_rows = cursor_gold.fetchall()
test_entries = [dict(zip(columns, row)) for row in test_rows]

# Ensure template name has .j2 extension for loading
template_file = template_name if template_name.endswith('.j2') else f"{template_name}.j2"

# Load template
env = Environment(loader=FileSystemLoader(f'{ROOT_PATH}/data/templates/{strategy}'))
template = env.get_template(template_file)

# Apply test_limit if specified
if test_limit is not None:
    test_entries = test_entries[:test_limit]

# train_skeleton_data = create_skeleton_dataset(train_entries)
train_skeleton_data = None

# Create datasets
train_data = create_dataset(train_entries, template, strategy=strategy, skeleton_dataset=train_skeleton_data)
valid_data = create_dataset(valid_entries, template, strategy=strategy)
test_data = create_dataset(test_entries, template, strategy=strategy)

train_sql_data = create_sql_dataset(train_entries)
valid_sql_data = create_sql_dataset(valid_entries)
test_sql_data = create_sql_dataset(test_entries)

# Write to JSONL files
def write_jsonl(data, filename):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')

def write_sql(data, filename):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, "w", encoding="utf-8") as f:
        for query in data:
            f.write(query + "\n")

folder_prefix = f"{ROOT_PATH}/data/training/{strategy}/{template_name.removesuffix('.j2')}/"
print(folder_prefix)
# write_jsonl(train_data, folder_prefix+'train.jsonl')
# write_jsonl(test_data, folder_prefix+'test.jsonl')
# write_jsonl(valid_data, folder_prefix+'valid.jsonl')

# write_sql(train_sql_data, folder_prefix+'train.sql')
# write_sql(test_sql_data, folder_prefix+'test.sql')
# write_sql(valid_sql_data, folder_prefix+'valid.sql')

# Close database connection
conn_gold.close()

# print(f"Dataset split and saved to {folder_prefix}: train ({len(train_data)} lines), test ({len(test_data)} lines), valid ({len(valid_data)} lines)")

Filtering dataset by difficulties: easy
Limiting test dataset to 10 records
/Users/atissera/Developer/repos/unrc-cs-thesis/data/training/nl2SQL/template_00/


In [29]:
len(train_entries)

1694

In [51]:
# # Better Approaches for Question Similarity
# import numpy as np
# import pandas as pd
# from collections import defaultdict, Counter
# import re
# from itertools import combinations
# import matplotlib.pyplot as plt
# import seaborn as sns

# def load_skeleton_questions(file_path):
#     """Load skeletons and questions from file (each line has skeleton \t query)"""
#     skeletons, queries = [], []
#     with open(file_path, 'r', encoding='utf-8') as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             skeleton, query = line.split('\t', 1)  # split only once
#             skeletons.append(skeleton.strip())
#             queries.append(query.strip())
#     return skeletons, queries

# # Load
# skeleton_file = f"{ROOT_PATH}/data/training/nl2SQL/template_00/skeleton.sql"
# skeletons, queries = load_skeleton_questions(skeleton_file)

# print(f"Loaded {len(skeletons)} skeletons and {len(queries)} queries")
# print("Sample:")
# for i in range(min(5, len(skeletons))):
#     print(f"{i+1}: {skeletons[i]}  ->  {queries[i]}")

In [52]:
# question = """from which countries are players who make more than 1200000 from?"""
# query = """select distinct country from player where salary > 1200000"""
# schema = """["club(Club_ID, Name, Manager, Captain, Manufacturer, Sponsor)", "player(Player_ID, Name, Country, Earnings, Events_number, Wins_count, Club_ID)"]"""



In [53]:
# train_skeleton_data
# skeleton_dataset, sql_queries = [], []
# for line in train_skeleton_data:
#     line = line.strip()
#     skeleton, query = line.split('\t', 1)
#     skeleton_dataset.append(skeleton.strip())
#     sql_queries.append(query.strip())

=== TESTING WITH YOUR ORIGINAL QUERY ===
Original question: what are the names of players in ascending order of wins count?
Skeleton: what are the <mask> of <mask> in ascending order of wins count ?

Top 3 similar skeletons to: 'what are the <mask> of <mask> in ascending order of wins count ?'
1. Score: 0.991 - Skeleton Question 53: what are the dates of <mask> in descending order of <mask> ?
1. Score: 0.991 - Full Question 53: what are the dates of publications in descending order of price?
1. Score: 0.991 - Query 53: select publication_date from publication order by price desc
2. Score: 0.987 - Skeleton Question 309: what are the birthdays of <mask> in ascending order of <mask> ?
2. Score: 0.987 - Full Question 309: what are the birthdays of people in ascending order of height?
2. Score: 0.987 - Query 309: select birth_date from people order by height asc
3. Score: 0.985 - Skeleton Question 682: list the <mask> of <mask> in ascending order of years working .
3. Score: 0.985 - Full Qu

'what are the dates of publications in descending order of price?\nselect publication_date from publication order by price desc\nwhat are the birthdays of people in ascending order of height?\nselect birth_date from people order by height asc\nlist the names of journalists in ascending order of years working.\nselect name from journalist order by years_working asc\n'

'what are the dates of publications in descending order of price?\nselect publication_date from publication order by price desc\nwhat are the birthdays of people in ascending order of height?\nselect birth_date from people order by height asc\nlist the names of journalists in ascending order of years working.\nselect name from journalist order by years_working asc\n'

In [25]:
# Apple Silicon Optimized Versions of get_similar_skeletons
import numpy as np
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import time
from functools import partial

# Check Apple Silicon optimizations
print("spaCy GPU acceleration:", spacy.prefer_gpu())
print("Available CPU cores:", mp.cpu_count())

# Check if thinc-apple-ops is properly installed and working
try:
    import thinc
    print("✅ thinc version:", thinc.__version__)
    
    # Check if Apple ops are available
    try:
        from thinc.api import get_current_ops
        ops = get_current_ops()
        print("✅ Current ops backend:", ops.name)
        print("✅ Apple Silicon optimizations:", "thinc-apple-ops" in str(ops))
    except Exception as e:
        print("⚠️  Could not check ops backend:", str(e))
        
except ImportError:
    print("❌ thinc not found")

# Note: spacy.prefer_gpu() returns False on Apple Silicon because
# Apple Silicon doesn't use traditional GPU acceleration like CUDA
# Instead, it uses CPU optimizations through the Accelerate framework
print("\n📝 Note: spacy.prefer_gpu() = False is CORRECT on Apple Silicon")
print("   Apple Silicon uses CPU optimizations, not GPU acceleration")
print("   The thinc-apple-ops package provides native Apple Silicon optimizations")


spaCy GPU acceleration: False
Available CPU cores: 10
✅ thinc version: 8.3.6
✅ Current ops backend: apple
✅ Apple Silicon optimizations: False

📝 Note: spacy.prefer_gpu() = False is CORRECT on Apple Silicon
   Apple Silicon uses CPU optimizations, not GPU acceleration
   The thinc-apple-ops package provides native Apple Silicon optimizations


In [12]:
# Version 1: Vectorized Similarity Computation
def get_similar_skeletons_vectorized(query_skeleton, train_skeleton_data, top_k=3):
    """
    Optimized version using vectorized operations and numpy for Apple Silicon
    """
    query_doc = nlp(query_skeleton)
    query_vector = query_doc.vector
    
    # Pre-compute all skeleton vectors in batch
    skeleton_vectors = []
    skeleton_info = []
    
    for i, line in enumerate(train_skeleton_data):
        line = line.strip()
        skeleton_question, full_question, sql_query = line.split('\t', 2)
        skeleton_doc = nlp(skeleton_question)
        skeleton_vectors.append(skeleton_doc.vector)
        skeleton_info.append((i, skeleton_question, full_question, sql_query))
    
    # Convert to numpy arrays for vectorized computation
    query_vector = np.array(query_vector)
    skeleton_vectors = np.array(skeleton_vectors)
    
    # Vectorized cosine similarity computation
    # Using numpy's optimized dot product and norm operations
    dot_products = np.dot(skeleton_vectors, query_vector)
    query_norm = np.linalg.norm(query_vector)
    skeleton_norms = np.linalg.norm(skeleton_vectors, axis=1)
    
    similarities = dot_products / (query_norm * skeleton_norms)
    
    # Create results with similarity scores
    results = []
    for i, similarity in enumerate(similarities):
        idx, skeleton_question, full_question, sql_query = skeleton_info[i]
        results.append((idx, similarity, skeleton_question, full_question, sql_query))
    
    # Sort by similarity (descending) and return top-k
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]


In [19]:
# Version 3: Hybrid Vectorized + Parallel Processing
def get_similar_skeletons_hybrid(query_skeleton, train_skeleton_data, top_k=3, n_workers=None):
    """
    Best of both worlds: vectorized operations + parallel processing
    """
    if n_workers is None:
        n_workers = min(mp.cpu_count(), 4)  # Limit to 4 workers to avoid overhead
    
    query_doc = nlp(query_skeleton)
    query_vector = np.array(query_doc.vector)
    
    # Split data into chunks for parallel processing
    chunk_size = len(train_skeleton_data) // n_workers
    chunks = []
    
    for i in range(n_workers):
        start_idx = i * chunk_size
        end_idx = start_idx + chunk_size if i < n_workers - 1 else len(train_skeleton_data)
        chunk_data = train_skeleton_data[start_idx:end_idx]
        chunks.append((query_vector, chunk_data, start_idx))
    
    def process_chunk_vectorized(args):
        query_vector, chunk_data, start_offset = args
        
        # Process chunk with vectorized operations
        skeleton_vectors = []
        skeleton_info = []
        
        for i, line in enumerate(chunk_data):
            line = line.strip()
            skeleton_question, full_question, sql_query = line.split('\t', 2)
            skeleton_doc = nlp(skeleton_question)
            skeleton_vectors.append(skeleton_doc.vector)
            skeleton_info.append((start_offset + i, skeleton_question, full_question, sql_query))
        
        if not skeleton_vectors:
            return []
        
        # Vectorized similarity computation
        skeleton_vectors = np.array(skeleton_vectors)
        dot_products = np.dot(skeleton_vectors, query_vector)
        query_norm = np.linalg.norm(query_vector)
        skeleton_norms = np.linalg.norm(skeleton_vectors, axis=1)
        similarities = dot_products / (query_norm * skeleton_norms)
        
        # Create results
        results = []
        for i, similarity in enumerate(similarities):
            idx, skeleton_question, full_question, sql_query = skeleton_info[i]
            results.append((idx, similarity, skeleton_question, full_question, sql_query))
        
        return results
    
    # Process chunks in parallel
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        chunk_results = list(executor.map(process_chunk_vectorized, chunks))
    
    # Flatten and sort results
    all_results = []
    for chunk_result in chunk_results:
        all_results.extend(chunk_result)
    
    all_results.sort(key=lambda x: x[1], reverse=True)
    return all_results[:top_k]


In [20]:
def get_few_shot(question, schema, train_skeleton_data=None):
    skeleton_query = get_question_skeleton(question, schema)
    similar_skeletons = get_similar_skeletons_hybrid(skeleton_query, train_skeleton_data, top_k=3)
    few_shot = ""
    for i, (idx, score, skeleton_question, full_question, sql_query) in enumerate(similar_skeletons, 1):
        few_shot += f"{full_question}\n{sql_query}\n"
    return few_shot

In [21]:
# Test with your original query
print("=== TESTING WITH YOUR ORIGINAL QUERY ===")

# Your original question
question = """what are the names of players in ascending order of wins count?"""
schema = """["club(Club_ID, Name, Manager, Captain, Manufacturer, Sponsor)", "player(Player_ID, Name, Country, Earnings, Events_number, Wins_count, Club_ID)"]"""

# # Create skeleton
# skeleton_query = get_question_skeleton(question, schema)
# print(f"Original question: {question}")
# print(f"Skeleton: {skeleton_query}")

# # Find similar skeletons
# print(f"\nTop 3 similar skeletons to: '{skeleton_query}'")
# similar_skeletons = get_similar_skeletons_vectorized(skeleton_query, train_skeleton_data, top_k=3)

# few_shot = ""
# for i, (idx, score, skeleton_question, full_question, sql_query) in enumerate(similar_skeletons, 1):
#     print(f"{i}. Score: {score:.3f} - Skeleton Question {idx}: {skeleton_question}")
#     print(f"{i}. Score: {score:.3f} - Full Question {idx}: {full_question}")
#     print(f"{i}. Score: {score:.3f} - Query {idx}: {sql_query}")
#     few_shot += f"{full_question}\n{sql_query}\n"
# few_shot

get_few_shot(question, schema, train_skeleton_data)

=== TESTING WITH YOUR ORIGINAL QUERY ===


AttributeError: Can't get local object 'get_similar_skeletons_hybrid.<locals>.process_chunk_vectorized'

In [24]:
# Benchmarking and Performance Comparison
def benchmark_function(func, query_skeleton, train_skeleton_data, top_k=3, n_runs=3):
    """Benchmark a function and return average execution time"""
    times = []
    for _ in range(n_runs):
        start_time = time.time()
        result = func(query_skeleton, train_skeleton_data, top_k)
        end_time = time.time()
        times.append(end_time - start_time)
    
    avg_time = sum(times) / len(times)
    return avg_time, result

# Test query and data
test_query = "what are the <mask> of <mask> in ascending order of wins count ?"
print("=== PERFORMANCE BENCHMARKING ===")
print(f"Test query: {test_query}")
print(f"Training data size: {len(train_skeleton_data)} entries")
print(f"Available CPU cores: {mp.cpu_count()}")
print()

# Benchmark all versions
functions_to_test = [
    ("Original", get_similar_skeletons),
    ("Vectorized", get_similar_skeletons_vectorized),
    ("Hybrid", get_similar_skeletons_hybrid)
]

results = {}
for name, func in functions_to_test:
    try:
        avg_time, result = benchmark_function(func, test_query, train_skeleton_data)
        results[name] = avg_time
        print(f"{name:12}: {avg_time:.4f}s")
        print(f"  Top result similarity: {result[0][1]:.4f}")
    except Exception as e:
        print(f"{name:12}: ERROR - {str(e)}")

print()
print("=== SPEEDUP COMPARISON ===")
if "Original" in results:
    baseline = results["Original"]
    for name, time_taken in results.items():
        if name != "Original":
            speedup = baseline / time_taken
            print(f"{name:12}: {speedup:.2f}x faster")


=== PERFORMANCE BENCHMARKING ===
Test query: what are the <mask> of <mask> in ascending order of wins count ?
Training data size: 1694 entries
Available CPU cores: 10

Original    : 1.6780s
  Top result similarity: 0.9906
Vectorized  : 1.6427s
  Top result similarity: 0.9906
Hybrid      : ERROR - Can't get local object 'get_similar_skeletons_hybrid.<locals>.process_chunk_vectorized'

=== SPEEDUP COMPARISON ===
Vectorized  : 1.02x faster


In [ ]:
# Version 4: Batch Processing for Multiple Queries
def get_similar_skeletons_batch(query_skeletons, train_skeleton_data, top_k=3):
    """
    Process multiple queries in batch for maximum efficiency
    Optimized for Apple Silicon with vectorized operations
    """
    # Pre-compute all training skeleton vectors once
    print("Pre-computing training skeleton vectors...")
    skeleton_vectors = []
    skeleton_info = []
    
    for i, line in enumerate(train_skeleton_data):
        line = line.strip()
        skeleton_question, full_question, sql_query = line.split('\t', 2)
        skeleton_doc = nlp(skeleton_question)
        skeleton_vectors.append(skeleton_doc.vector)
        skeleton_info.append((i, skeleton_question, full_question, sql_query))
    
    skeleton_vectors = np.array(skeleton_vectors)
    skeleton_norms = np.linalg.norm(skeleton_vectors, axis=1)
    
    # Process all queries
    all_results = []
    for query_skeleton in query_skeletons:
        query_doc = nlp(query_skeleton)
        query_vector = np.array(query_doc.vector)
        query_norm = np.linalg.norm(query_vector)
        
        # Vectorized similarity computation for this query
        dot_products = np.dot(skeleton_vectors, query_vector)
        similarities = dot_products / (query_norm * skeleton_norms)
        
        # Create results for this query
        query_results = []
        for i, similarity in enumerate(similarities):
            idx, skeleton_question, full_question, sql_query = skeleton_info[i]
            query_results.append((idx, similarity, skeleton_question, full_question, sql_query))
        
        # Sort and get top-k
        query_results.sort(key=lambda x: x[1], reverse=True)
        all_results.append(query_results[:top_k])
    
    return all_results

# Example usage for batch processing
def process_multiple_queries_batch(queries, train_skeleton_data, top_k=3):
    """Wrapper function for batch processing multiple queries"""
    return get_similar_skeletons_batch(queries, train_skeleton_data, top_k)


False